# **Original prompt**
One must have a mind of winter

To regard the frost and the boughs

Of the pine-trees crusted with snow;

And have been cold a long time

To behold the junipers shagged with ice,

The spruces rough in the distant glitter

Of the January sun; and not to think

Of any misery in the sound of the wind,

In the sound of a few leaves,

Which is the sound of the land

Full of the same wind

That is blowing in the same bare place

For the listener, who listens in the snow,

And, nothing himself, beholds

Nothing that is not there and the nothing that is.

# **Substantial nouns removed**
One must have a ___ of ___

To regard the ___ and the ___

Of the ___ crusted with ___;

And have been ___ a long ___

To behold the ___ shagged with ___,

The ___ rough in the distant ___

Of the January ___; and not to think

Of any misery in the ___ of the ___,

In the ___ of a few ___,

Which is the ___ of the ___

Full of the same ___

That is blowing in the same bare ___

For the ___, who listens in the ___,

And, ___ himself, beholds

___ that is not there and the ___ that is.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2" # Using gpt2 as a language model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

poem = """One must have a ___ of ___
To regard the ___ and the ___
Of the ___ crusted with ___;
And have been ___ a long ___
To behold the ___ shagged with ___,
The ___ rough in the distant ___
Of the January ___; and not to think
Of any misery in the ___ of the ___,
In the ___ of a few ___,
Which is the ___ of the ___
Full of the same ___
That is blowing in the same bare ___
For the ___, who listens in the ___,
And, ___ himself, beholds
___ that is not there and the ___ that is."""

# Set pad_token_id to eos_token_id if it's not set, which is common for GPT-like models.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
def fill_all_blanks(poem_with_blanks, model, tokenizer, n_word_index):
    filled_poem = poem_with_blanks
    blank_marker = "___"

    while blank_marker in filled_poem:
        # Find the text before the current blank
        parts = filled_poem.split(blank_marker, 1) # Split only at the first blank
        text_before_current_blank = parts[0].strip()

        # Encode the text before the blank
        input_ids = tokenizer.encode(text_before_current_blank, return_tensors='pt')

        # Get model predictions
        with torch.no_grad():
            outputs = model(input_ids)
            predictions = outputs.logits

        # Get the nth most likely word
        last_token_logits = predictions[0, -1, :]
        sorted_logits, sorted_indices = torch.sort(last_token_logits, descending=True)

        if n_word_index - 1 < len(sorted_indices):
            nth_likely_token_id = sorted_indices[n_word_index - 1].item()
            predicted_word = tokenizer.decode([nth_likely_token_id]).strip()
        else:
            predicted_word = "[UNK]" # Fallback if n is too large or vocab is small

        # Replace only the first blank with the predicted word
        filled_poem = filled_poem.replace(blank_marker, predicted_word, 1)

    return filled_poem

In [3]:
nth_word_index_for_prediction = 7

filled_poem_final = fill_all_blanks(poem, model, tokenizer, nth_word_index_for_prediction)
print("\n--- Filled Poem ---")
print(filled_poem_final)


--- Filled Poem ---
One must have a little of it
To regard the way and the form
Of the man crusted with blood;
And have been to a long day
To behold the body shagged with bones,
The flesh rough in the distant ground
Of the January wind; and not to think
Of any misery in the place of the earth,
In the sight of a few thousand,
Which is the greatest of the Gods
Full of the same Gods
That is blowing in the same bare sky
For the earth, who listens in the darkness,
And, behold himself, beholds
in that is not there and the other that is.
